<a href="https://colab.research.google.com/github/Grenki-with-cheese/2213935-CN5004-exercises/blob/main/notebook06_Recovered_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setting up SD v1.4 with Erased checkpoint (see notebook04, notebook04)

In [1]:
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \ numpy==1.26.4 "scipy<1.14" -q
print("Install complete.")

Install complete.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MyDissertationCN6000')
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
assert torch.cuda.is_available(), "Need a GPU runtime (A100 or L4)."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [3]:
from pathlib import Path
from diffusers import StableDiffusionPipeline
import safetensors.torch

# Use the path to your ESD checkpoint
CHECKPOINT_PATH = Path("checkpoints/esd_vangogh_2026-05-06.safetensors")
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.set_progress_bar_config(disable=True)

state = safetensors.torch.load_file(str(CHECKPOINT_PATH))
result = pipe.unet.load_state_dict(state, strict=False)
print(f"Loaded {len(state)} cross-attention keys.")
print(f"Missing keys (frozen layers, expected): {len(result.missing_keys)}")
assert len(result.unexpected_keys) == 0, f"Unexpected keys: {result.unexpected_keys[:5]}"

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://gi

Loaded 80 cross-attention keys.
Missing keys (frozen layers, expected): 606


Generating using recovered prompts (from notebook05)

In [4]:
import json

SEEDS = [42, 1337, 2026, 7777, 2213935]
GUIDANCE = 9.0
STEPS = 50
NEGATIVE = ("photograph, photo, realistic, 3d render, cgi, blurry, deformed face, "
            "extra fingers, ugly, low quality, oversaturated, anime, cartoon, "
            "watermark, signature, text")

def generate_set(prompts, seeds, output_dir):
    """Generate one image per (prompt, seed) pair, saved with paired filename."""
    output_dir.mkdir(parents=True, exist_ok=True)
    log = []
    total = len(prompts) * len(seeds)
    count = 0
    for prompt_idx, prompt in enumerate(prompts):
        for seed in seeds:
            generator = torch.Generator("cuda").manual_seed(seed)
            image = pipe(
                prompt,
                negative_prompt=NEGATIVE,
                generator=generator,
                num_inference_steps=STEPS,
                guidance_scale=GUIDANCE,
            ).images[0]
            filename = f"p{prompt_idx:03d}_s{seed:06d}.png"
            image.save(output_dir / filename)
            log.append({
                "prompt_idx": prompt_idx,
                "prompt": prompt,
                "seed": seed,
                "filename": filename,
            })
            count += 1
            if count % 10 == 0 or count == total:
                print(f"  {count}/{total}")
    return log

recovered = json.loads(Path("prompts/recovered_prompts.json").read_text())
assert len(recovered) == 10, f"Expected 10 recovered prompts, got {len(recovered)}"

print(f"Generating {len(recovered) * len(SEEDS)} recovered images...")
log_r = generate_set(recovered, SEEDS, Path("outputs/recovered"))

Path("logs").mkdir(exist_ok=True)
Path("logs/seeds_recovered.json").write_text(
    json.dumps({"recovered": log_r}, indent=2)
)
print(f"\nSaved seed log to logs/seeds_recovered.json")

Generating 50 recovered images...
  10/50
  20/50
  30/50
  40/50
  50/50

Saved seed log to logs/seeds_recovered.json


Verifying image count

In [5]:
n_recovered = len(list(Path("outputs/recovered").glob("*.png")))
assert n_recovered == 50, f"Recovered count mismatch: expected 50, got {n_recovered}"
print(f"Recovered generation complete: {n_recovered} images")

Recovered generation complete: 50 images
